# 🚀 Qwen 2.5 Solver — 24hr Training on Colab

**One-click training pipeline using Unsloth + our custom pipeline.**

- ✅ Works on **free Colab T4 (16GB)** → trains Qwen 2.5 7B
- ✅ Works on **Colab Pro A100 (40GB)** → trains Qwen 2.5 14B
- ✅ Saves every 100 steps so you can resume if disconnected
- ✅ Auto-exports to GGUF + Ollama Modelfile at the end

---

## Step 0: Check GPU

In [ ]:
!nvidia-smi

## Step 1: Install dependencies (~3 min)

In [ ]:
!pip install -q unsloth transformers datasets trl peft accelerate bitsandbytes xformers
!pip install -q flash-attn --no-build-isolation 2>/dev/null || echo "flash-attn skipped (optional)"
print("✅ Dependencies installed")

## Step 2: Clone the pipeline

In [ ]:
!git clone https://github.com/ghaith012x-collab/cs.git /content/pipeline-repo 2>/dev/null || (cd /content/pipeline-repo && git pull)
import sys; sys.path.insert(0, '/content/pipeline-repo')
%cd /content/pipeline-repo
print("✅ Pipeline ready")

## Step 3: Collect training data

In [ ]:
# Download the best open-source datasets from HuggingFace
# This gets you LeetCode, NuminaMath, OpenOrca, CodeAlpaca, Magicoder
from pipeline.datasets import download_all, merge_datasets
import glob

download_all(output_dir='data/hf', max_per_dataset=3000)
files = sorted(glob.glob('data/hf/*.jsonl'))
merge_datasets(files, output='data/train_merged.jsonl')

!wc -l data/train_merged.jsonl
print('✅ Data ready')

## Step 4: TRAIN — this is the 24hr run

**Auto-detects GPU and picks optimal settings.**

| GPU | Model | LoRA r | Batch | Seq Len | Epochs |
|---|---|---|---|---|---|
| A100 | qwen2.5-14b | 64 | 4 | 4096 | 3 |
| T4/V100 | qwen2.5-7b | 32 | 2 | 2048 | 3 |

In [ ]:
import torch

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name} ({vram_gb:.0f}GB VRAM)')

if vram_gb >= 35:  # A100
    model = 'qwen2.5-14b'
    lora_r = 64
    batch = 4
    seq_len = 4096
elif vram_gb >= 20:  # 3090/4090/A6000
    model = 'qwen2.5-7b'
    lora_r = 64
    batch = 4
    seq_len = 4096
else:  # T4 (16GB)
    model = 'qwen2.5-7b'
    lora_r = 32
    batch = 2
    seq_len = 2048

print(f'→ Model: {model}, LoRA r={lora_r}, batch={batch}, seq_len={seq_len}')

In [ ]:
from pipeline.train import run_training
from pipeline.config import TrainingConfig

config = TrainingConfig(
    data_path='data/train_merged.jsonl',
    output_dir='output/solver',
    model_key=model,
    ollama_model_name='solver-v1',
    lora_r=lora_r,
    lora_alpha=lora_r * 2,
    per_device_train_batch_size=batch,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    num_train_epochs=3,
    max_seq_length=seq_len,
    warmup_ratio=0.1,
    weight_decay=0.01,
    packing=True,
    save_steps=100,
    logging_steps=10,
    seed=42,
)

adapter_path = run_training(config)
print(f'✅ Training complete → {adapter_path}')

## Step 5: Evaluate

In [ ]:
from pipeline.evaluate import run_evaluation
run_evaluation(config)

## Step 6: Export to GGUF

In [ ]:
from pipeline.export import run_export
run_export(config)

# Download the GGUF file
from google.colab import files
import glob
for f in glob.glob('output/solver/ollama-model/*.gguf'):
    files.download(f)
    print(f'✅ Downloaded: {f}')

for f in glob.glob('output/solver/ollama-model/Modelfile'):
    files.download(f)
    print(f'✅ Downloaded: Modelfile')

---
## 🏠 After download — run locally with Ollama:

```bash
# 1. Install Ollama: curl -fsSL https://ollama.com/install.sh | sh

# 2. Put the .gguf and Modelfile in a folder, then:
ollama create solver-v1 -f Modelfile

# 3. Run!
ollama run solver-v1
```

---
### 🔄 Resume if Colab disconnects

Colab free tier disconnects after ~4-6 hours. The pipeline saves checkpoints every 100 steps to `output/solver/`. To resume:

1. Reconnect and re-run Steps 0-2
2. In Step 4, change the config to pick up from the last checkpoint:
   ```python
   config.output_dir = 'output/solver'  # same dir, HF Trainer auto-resumes
   ```
3. Re-run Step 4 — it picks up where it left off!

**Pro tip:** Mount Google Drive to persist checkpoints:
```python
from google.colab import drive
drive.mount('/content/drive')
config.output_dir = '/content/drive/MyDrive/solver-training'
```